In [2]:
import warnings

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold as KFold

from my_useful_functions import (
    calculate_performance_statistical_parity,
    calculate_performance_equalized_odds,
    calculate_performance_equal_opportunity,
    calculate_performance_predictive_parity,
    calculate_performance_predictive_equality,
    calculate_performance_treatment_equality,
)
from compute_abroca import compute_abroca

warnings.filterwarnings("ignore")

from pathlib import Path

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



In [3]:
def _postprocess_split(
    df: pd.DataFrame,
    protected_attribute: str,
    class_label: str,
    filename: str,
    majority_group_name: str,
    minority_group_name: str,
):

    if class_label not in df.columns:
        raise ValueError(
            f"class_label '{class_label}' not found in dataframe columns: {df.columns.tolist()}"
        )

    cols = [c for c in df.columns if c != class_label] + [class_label]
    df = df[cols]

    p_Group = 0

    for col in df.select_dtypes(include=["object", "category"]).columns:
        le = preprocessing.LabelEncoder()
        le.fit(df[col])

        if col == protected_attribute:
            if majority_group_name in le.classes_:
                p_Group = int(np.where(le.classes_ == majority_group_name)[0][0])

        df[col] = le.transform(df[col])

    X = df.iloc[:, :-1].copy()
    y = df.iloc[:, -1].copy()

    y = y.replace(0, -1)

    feature_names = X.columns.tolist()
    if protected_attribute not in feature_names:
        raise ValueError(
            f"protected_attribute '{protected_attribute}' not found in features: {feature_names}"
        )
    sa_index = feature_names.index(protected_attribute)

    return (
        X,
        y,
        sa_index,
        p_Group,
        protected_attribute,
        filename,
        majority_group_name,
        minority_group_name,
    )


In [4]:
def recode_original(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:

    # -------- Compas-Original --------
    if dataset_name == "Compas-Original":
        if not {"juv_fel_count", "juv_misd_count", "juv_other_count"}.issubset(df.columns):
            raise ValueError("Compas-Original is missing juvenile count columns.")

        df["juv_crime"] = (
            df["juv_fel_count"] + df["juv_misd_count"] + df["juv_other_count"]
        )

        selected_cols = [
            "race",
            "sex",
            "two_year_recid",
            "score_text",
            "priors_count",
            "age_cat",
            "v_score_text",
            "juv_crime",
            "c_charge_degree",
        ]
        missing = [c for c in selected_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Compas-Original is missing columns: {missing}")
        df = df[selected_cols]

    # -------- Compas-Violent-Original --------
    elif dataset_name == "Compas-Violent-Original":
        if not {"juv_fel_count", "juv_misd_count", "juv_other_count"}.issubset(df.columns):
            raise ValueError("Compas-Violent-Original is missing juvenile count columns.")

        df["juv_crime"] = (
            df["juv_fel_count"] + df["juv_misd_count"] + df["juv_other_count"]
        )

        selected_cols = [
            "race",
            "sex",
            "two_year_recid",
            "score_text",
            "priors_count",
            "age_cat",
            "v_score_text",
            "juv_crime",
            "c_charge_degree",
        ]
        missing = [c for c in selected_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Compas-Violent-Original is missing columns: {missing}")
        df = df[selected_cols]

    return df


def add_binary_race(df, dataset_name: str, new_col="race_bin"):
    if "race" not in df.columns:
        raise ValueError(f"{dataset_name}: expected column 'race' not found")

    if dataset_name.endswith("Original"):
        df[new_col] = np.where(df["race"] == "Caucasian", "Caucasian", "Non-Caucasian")
    else:
        df[new_col] = np.where(df["race"] == 2, "Caucasian", "Non-Caucasian")

    return df



In [8]:
def load_dataset(dataset_name: str):
    CONFIG = {
        # --------- LAW ----------
        "Law-Original": {
            "path": "data/law_school_clean.csv",
            "protected_attribute": "male",
            "class_label": "pass_bar",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Law-Original.abroca.pdf",
        },
        "Law-Synthetic": {
            "path": "data/law_synthetic.csv",
            "protected_attribute": "male",
            "class_label": "pass_bar",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Law-Synthetic.abroca.pdf",
        },
        "Law-Debiased": {
            "path": "data/law_debiased.csv",
            "protected_attribute": "male",
            "class_label": "pass_bar",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Law-Debiased.abroca.pdf",
        },
        "Law-FLAI": {
            "path": "data/Law-Flai-new-synthetic_fair_data.csv",
            "protected_attribute": "male",
            "class_label": "pass_bar",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Law-FLAI.abroca.pdf",
        },
        "Law-DECAF": {
            "path": "data/law_decaf.csv",
            "protected_attribute": "male",
            "class_label": "pass_bar",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Law-DECAF.abroca.pdf",
        },

        # --------- ADULT ----------
        "Adult-Original": {
            "path": "data/adult-clean.csv",
            "protected_attribute": "gender",
            "class_label": "Class-label",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Adult-Original.abroca.pdf",
        },
        "Adult-Synthetic": {
            "path": "data/adult_synthetic.csv",
            "protected_attribute": "gender",
            "class_label": "Class-label",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Adult-Synthetic.abroca.pdf",
        },
        "Adult-Debiased": {
            "path": "data/adult_debiased.csv",
            "protected_attribute": "gender",
            "class_label": "Class-label",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Adult-Debiased.abroca.pdf",
        },
        "Adult-FLAI": {
            "path": "data/Adult-Flai-new-synthetic_fair_data.csv",
            "protected_attribute": "gender",
            "class_label": "Class-label",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Adult-FLAI.abroca.pdf",
        },
        "Adult-DECAF": {
            "path": "data/adult_decaf.csv",
            "protected_attribute": "gender",
            "class_label": "Class-label",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Adult-DECAF.abroca.pdf",
        },

        # --------- COMPAS (non-violent) ----------
        "Compas-Original": {
            "path": "data/compas-scores-two-years_clean.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Original.abroca.pdf",
        },
        "Compas-Synthetic": {
            "path": "data/compas_synthetic.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Synthetic.abroca.pdf",
        },
        "Compas-Debiased": {
            "path": "data/compas_debiased.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Debiased.abroca.pdf",
        },
        "Compas-FLAI": {
            "path": "data/Compas-Flai-new-synthetic_fair_data.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-FLAI.abroca.pdf",
        },
        "Compas-DECAF": {
            "path": "data/compas_decaf.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-DECAF.abroca.pdf",
        },

        # --------- COMPAS-VIOLENT ----------
        "Compas-Violent-Original": {
            "path": "data/compas-scores-two-years-violent_clean.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Violent-Original.abroca.pdf",
        },
        "Compas-Violent-Synthetic": {
            "path": "data/compas_violent_synthetic.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Violent-Synthetic.abroca.pdf",
        },
        "Compas-Violent-Debiased": {
            "path": "data/compas_violent_debiased.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Violent-Debiased.abroca.pdf",
        },
        "Compas-Violent-FLAI": {
            "path": "data/Compas-Violent-Flai-new-synthetic_fair_data.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Violent-FLAI.abroca.pdf",
        },
        "Compas-Violent-DECAF": {
            "path": "data/compas_violent_decaf.csv",
            "protected_attribute": "sex",
            "class_label": "two_year_recid",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            # "protected_attribute": "race_bin",
            # "majority_group_name": "Caucasian",
            # "minority_group_name": "Non-Caucasian",
            "filename": "Ada.Compas-Violent-DECAF.abroca.pdf",
        },

        # --------- DIABETES ----------
        "Diabetes-Original": {
            "path": "data/diabetes-clean.csv",
            "protected_attribute": "gender",
            "class_label": "readmitted",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Diabetes-Original.abroca.pdf",
        },
        "Diabetes-Synthetic": {
            "path": "data/diabetes_synthetic.csv",
            "protected_attribute": "gender",
            "class_label": "readmitted",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Diabetes-Synthetic.abroca.pdf",
        },
        "Diabetes-Debiased": {
            "path": "data/diabetes_debiased.csv",
            "protected_attribute": "gender",
            "class_label": "readmitted",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Diabetes-Debiased.abroca.pdf",
        },
        "Diabetes-FLAI": {
            "path": "data/diabetes_flai.csv",
            "protected_attribute": "gender",
            "class_label": "readmitted",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Diabetes-FLAI.abroca.pdf",
        },
        "Diabetes-DECAF": {
            "path": "data/diabetes_decaf.csv",
            "protected_attribute": "gender",
            "class_label": "readmitted",
            "majority_group_name": "Male",
            "minority_group_name": "Female",
            "filename": "Ada.Diabetes-DECAF.abroca.pdf",
        },

        # --------- DUTCH ----------
        "Dutch-Original": {
            "path": "data/dutch.csv",
            "protected_attribute": "sex",
            "class_label": "occupation",
            "majority_group_name": "male",
            "minority_group_name": "female",
            "filename": "Ada.Dutch-Original.abroca.pdf",
        },
        "Dutch-Synthetic": {
            "path": "data/dutch_synthetic.csv",
            "protected_attribute": "sex",
            "class_label": "occupation",
            "majority_group_name": "male",
            "minority_group_name": "female",
            "filename": "Ada.Dutch-Synthetic.abroca.pdf",
        },
        "Dutch-Debiased": {
            "path": "data/dutch_debiased.csv",
            "protected_attribute": "sex",
            "class_label": "occupation",
            "majority_group_name": "male",
            "minority_group_name": "female",
            "filename": "Ada.Dutch-Debiased.abroca.pdf",
        },
        "Dutch-FLAI": {
            "path": "data/Dutch-Flai-new-synthetic_fair_data.csv",
            "protected_attribute": "sex",
            "class_label": "occupation",
            "majority_group_name": "male",
            "minority_group_name": "female",
            "filename": "Ada.Dutch-FLAI.abroca.pdf",
        },
        "Dutch-DECAF": {
            "path": "data/dutch_decaf.csv",
            "protected_attribute": "sex",
            "class_label": "occupation",
            "majority_group_name": "male",
            "minority_group_name": "female",
            "filename": "Ada.Dutch-DECAF.abroca.pdf",
        },
    }

    if dataset_name not in CONFIG:
        raise ValueError(f"Unknown dataset_name '{dataset_name}'")

    cfg = CONFIG[dataset_name]

    df = pd.read_csv(cfg["path"])

    if dataset_name.endswith("Original"):
        df = recode_original(df, dataset_name)

    # if dataset_name.startswith("Compas"):
    #     df = add_binary_race(df, dataset_name, new_col="race_bin")

    return _postprocess_split(
        df=df,
        protected_attribute=cfg["protected_attribute"],
        class_label=cfg["class_label"],
        filename=cfg["filename"],
        majority_group_name=cfg["majority_group_name"],
        minority_group_name=cfg["minority_group_name"],
    )


In [10]:
def run_experiment_mlp(
    X,
    y,
    sa_index,
    p_Group,
    protected_attribute,
    filename,
    majority_group_name,
    minority_group_name,
    dataset_name: str,
):

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    results = []

    for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
        # print(f"\n--- {dataset_name} | Fold {fold + 1}/10 ---")

        X_train_fold = X.iloc[train_index].copy()
        X_test_fold  = X.iloc[test_index].copy()
        y_train_fold = y.iloc[train_index].copy()
        y_test_fold  = y.iloc[test_index].copy()

        # ----- Model -----
        mlp = MLPClassifier(
            random_state=1,
            max_iter=300,
        )
        mlp.fit(X_train_fold, y_train_fold)

        y_predicts = mlp.predict(X_test_fold)

        if hasattr(mlp, "predict_proba"):
            y_pred_probs = mlp.predict_proba(X_test_fold)[:, 1]
        else:
            y_pred_probs = np.zeros_like(y_predicts, dtype=float)

        y_test_bin = (y_test_fold == 1).astype(int).values
        y_pred_bin = (y_predicts == 1).astype(int)

        # ----- Fairness & performance metrics -----

        sp_stats = calculate_performance_statistical_parity(
            X_test_fold.values, y_test_bin, y_pred_bin, sa_index, p_Group
        )
        acc  = sp_stats.get("accuracy", 0.0)
        bacc = sp_stats.get("balanced_accuracy", 0.0)
        sp   = sp_stats.get("fairness", 0.0)

        eo = calculate_performance_equal_opportunity(
            X_test_fold.values, y_test_bin, y_pred_bin, sa_index, p_Group
        ).get("fairness", 0.0)

        eod = calculate_performance_equalized_odds(
            X_test_fold.values, y_test_bin, y_pred_bin, y_pred_probs, sa_index, p_Group
        ).get("fairness", 0.0)

        pp = calculate_performance_predictive_parity(
            X_test_fold.values, y_test_bin, y_pred_bin, sa_index, p_Group
        ).get("fairness", 0.0)

        pe = calculate_performance_predictive_equality(
            X_test_fold.values, y_test_bin, y_pred_bin, sa_index, p_Group
        ).get("fairness", 0.0)

        te = calculate_performance_treatment_equality(
            X_test_fold.values, y_test_bin, y_pred_bin, sa_index, p_Group
        ).get("fairness", 0.0)

        # ----- ABROCA -----
        if dataset_name == "Dutch-DECAF":
            abroca = np.nan
        else:
            X_test_tmp = X_test_fold.copy()
            X_test_tmp["pred_proba"] = y_pred_probs
            X_test_tmp["true_label"] = y_test_fold.values

            abroca = compute_abroca(
                X_test_tmp,
                pred_col="pred_proba",
                label_col="true_label",
                protected_attr_col=protected_attribute,
                majority_protected_attr_val=p_Group,
                n_grid=10000,
                plot_slices=False,
                majority_group_name=majority_group_name,
                minority_group_name=minority_group_name,
                file_name=f"{dataset_name}_Fold{fold + 1}_" + filename,
            )

        results.append(
            {
                "Accuracy":            acc,
                "Balanced Accuracy":   bacc,
                "Statistical Parity":  sp,
                "Equal Opportunity":   eo,
                "Equalized Odds":      eod,
                "Predictive Parity":   pp,
                "Predictive Equality": pe,
                "Treatment Equality":  te,
                "ABROCA":              abroca,
            }
        )

    df_results = pd.DataFrame(results)
    avg_results = df_results.mean().round(4)

    print(f"\n===== Average Results Across Folds (MLP Model) | {dataset_name} =====")
    print(avg_results.to_frame(name="Average MLP (10-Fold)").T)

    avg_results_df = avg_results.reset_index()
    avg_results_df.columns = ["Fairness Metric", "MLP"]

    # Optional per-dataset CSV
    # out_name = f"{dataset_name}_mlp_results_new1.csv"
    # avg_results_df.to_csv(OUTPUT_DIR / out_name, index=False)
    # print(f"Saved per-dataset MLP results: {out_name}")

    return avg_results_df

In [ ]:
def run_all_datasets_mlp():
    dataset_names = [
        "Law-Original", "Law-Synthetic", "Law-Debiased", "Law-FLAI", "Law-DECAF",
        "Adult-Original", "Adult-Synthetic", "Adult-Debiased", "Adult-FLAI", "Adult-DECAF",
        "Compas-Original", "Compas-Synthetic", "Compas-Debiased", "Compas-FLAI", "Compas-DECAF",
        "Compas-Violent-Original", "Compas-Violent-Synthetic", "Compas-Violent-Debiased",
        "Compas-Violent-FLAI", "Compas-Violent-DECAF",
        "Diabetes-Original", "Diabetes-Synthetic", "Diabetes-Debiased",
        # "Diabetes-FLAI", 
        "Diabetes-DECAF",
        "Dutch-Original", "Dutch-Synthetic", "Dutch-Debiased",
        "Dutch-FLAI", "Dutch-DECAF",
        # "Compas-FLAI", "Compas-Violent-FLAI"
    ]

    all_results = []

    for name in dataset_names:
        print("\n===========================================")
        print(f"Running MLP on dataset: {name}")

        (
            X,
            y,
            sa_index,
            p_Group,
            protected_attribute,
            filename,
            majority_group_name,
            minority_group_name,
        ) = load_dataset(name)

        avg_df = run_experiment_mlp(
            X,
            y,
            sa_index,
            p_Group,
            protected_attribute,
            filename,
            majority_group_name,
            minority_group_name,
            dataset_name=name,
        )

        avg_df.insert(0, "Dataset", name)
        all_results.append(avg_df)

    combined = pd.concat(all_results, ignore_index=True)

    # Final combined CSV / Excel
    # combined.to_csv(OUTPUT_DIR / "Combined_MLP_results.csv", index=False)
    # If you prefer Excel:
    # combined.to_excel("Combined_MLP_results.xlsx", index=False)

    print("\nSaved Combined_MLP_results.csv with all datasets.")
    return combined

combined_mlp_results = run_all_datasets_mlp()
